<a href="https://colab.research.google.com/github/Sances84/files/blob/main/Algobit_algodoo_phunlet_extraction_program%2C_OnlyLine_Co_project_done_by_CEO_and_GEMINI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
import shutil

# Create /content if it doesn't exist
if not os.path.exists('/content'):
    os.makedirs('/content')

# Path to your Google Drive
drive_path = '/content/drive/MyDrive'

# Find and copy all .phz files
phz_files_found = []
for root, dirs, files in os.walk(drive_path):
    for file in files:
        if file.endswith('.phz'):
            source_path = os.path.join(root, file)
            destination_path = os.path.join('/content', file)
            shutil.copy2(source_path, destination_path)
            phz_files_found.append(destination_path)

if phz_files_found:
    print(f"Copied {len(phz_files_found)} .phz files to /content:")
    for f in phz_files_found:
        print(f"- {f}")
else:
    print("No .phz files found in Google Drive.")

No .phz files found in Google Drive.


# **WHAT YOU ARE NEED TO KNOW:**
# **PLEASE READ THIS!!!**
# **IMPORT THE PHZ (ALGODOO PHUNLET) FILES IN GOOGLE DRIVE IN PARENT DIRECTORY.**
# **THEN RUN IT**
# **PLEASE NOTE THAT THE PROGRAM SUPPORTS ONLY POLYGON OBJECTS**
# **CAN BE BUGGY, NOT RECOGNIZE POLYGONS**

In [ ]:
import zipfile
import os

def extract_phz(phz_filepath, extract_to_dir):
    if not os.path.exists(extract_to_dir):
        os.makedirs(extract_to_dir)

    try:
        with zipfile.ZipFile(phz_filepath, 'r') as zip_ref:
            zip_ref.extractall(extract_to_dir)
        print(f"Successfully extracted '{phz_filepath}' to '{extract_to_dir}'")
        return True
    except zipfile.BadZipFile:
        print(f"Error: '{phz_filepath}' is not a valid zip file. It might not be an Algodoo PHZ file or is corrupted.")
        return False
    except Exception as e:
        print(f"An error occurred while extracting '{phz_filepath}': {e}")
        return False

# Assuming you have at least one .phz file copied to /content
# You might need to adjust this to pick a specific file if you have many
if 'phz_files_found' in locals() and phz_files_found:
    first_phz_file = phz_files_found[0]
    extract_dir = os.path.join('/content', os.path.splitext(os.path.basename(first_phz_file))[0] + '_extracted')
    print(f"Attempting to extract the first found .phz file: {first_phz_file}")
    if extract_phz(first_phz_file, extract_dir):
        print(f"Contents of the extracted directory '{extract_dir}':")
        for root, dirs, files in os.walk(extract_dir):
            for name in files:
                print(os.path.join(root, name))
            for name in dirs:
                print(os.path.join(root, name) + '/')
else:
    print("No .phz files found to extract.")

After extracting, you'll typically find a text file (often named `scene.phz` or similar within the extracted folder) that describes the objects and properties of the Algodoo scene in a custom format. We can use Python to parse this file and extract the scene data.

# **try the examples:**
# **for glass vase v1 and more phz files can be found [here!!!](https://github.com/Sances84/files/)**
# **or run this code below to import those:**

In [ ]:
import requests
import os
repo_url = "https://api(.)github(.)com/repos/Sances84/files/contents/"
raw_base_url = "https://raw(.)githubusercontent(.)com/Sances84/files/main/"
def download_phz_files():
    response = requests.get(repo_url.replace("(.)", "."))
    if response.status_code == 200:
        files = response.json()
        for file in files:
            if file['name'].endswith('.phz'):
                file_name = file['name']
                download_url = raw_base_url.replace("(.)", ".") + file_name.replace(" ", "%20")

                print(f"downloading: {file_name}...")
                r = requests.get(download_url)
                with open(file_name, 'wb') as f:
                    f.write(r.content)
        print("\ndownloaded all the files of phz directly! check out files section")
    else:
        print("Error-no access to repo")
download_phz_files()

downloading: glass vase v1.phz...
downloading: glass vase v2.phz...
downloading: wooden vase v1.phz...
downloading: wooden vase v2.phz...

downloaded all the files of phz directly! check out files section


In [ ]:
import re
import os
from IPython.display import Image, display

# Function to parse the Algodoo .phn file
def parse_algodoo_phn(filepath):
    with open(filepath, 'r', encoding='utf-8') as f:
        content = f.read()

    parsed_data = {
        'FileInfo': {},
        'Polygons': []
    }

    # Parse FileVersion
    file_version_match = re.search(r'^// FileVersion (\d+)', content, re.MULTILINE)
    if file_version_match:
        parsed_data['FileVersion'] = int(file_version_match.group(1))

    # Parse FileInfo
    file_info_match = re.search(r'FileInfo -> \{(.*?)\};', content, re.DOTALL)
    if file_info_match:
        info_str = file_info_match.group(1)
        for match in re.finditer(r'(\w+)\s*=\s*(".*?"|\d+\.?\d*);', info_str):
            key = match.group(1)
            value = match.group(2)
            if value.startswith('"') and value.endswith('"'):
                parsed_data['FileInfo'][key] = value[1:-1]
            else:
                try:
                    parsed_data['FileInfo'][key] = int(value)
                except ValueError:
                    parsed_data['FileInfo'][key] = float(value)

    # Parse Scene.addPolygon blocks
    polygon_blocks = re.findall(r'Scene\\.addPolygon\s*\{(.*?)\}', content, re.DOTALL)
    for block_str in polygon_blocks:
        polygon_data = {}
        # Parse surfaces
        surfaces_match = re.search(r'surfaces\s*:=\s*\[\[\[(.*?)\]\]\];', block_str, re.DOTALL)
        if surfaces_match:
            points_raw = surfaces_match.group(1)
            # Find all [x, y] pairs
            points_list_str = re.findall(r'\[(-?\d+\.?\d*(?:e[+-]?\d+)?)\s*,\s*(-?\d+\.?\d*(?:e[+-]?\d+)?)\]', points_raw)
            polygon_data['surfaces'] = [[[float(x), float(y)] for x, y in points_list_str]] # Nested list as in example

        # Parse other properties
        for match in re.finditer(r'(\w+)\s*:=\s*(.*?);', block_str):
            key = match.group(1)
            value_str = match.group(2).strip()

            if key == 'surfaces': # Already handled
                continue

            # Try to parse value types
            if value_str.startswith('[') and value_str.endswith(']'):
                # This is a list. Could be numbers or booleans. e.g., [1.0, 0.5, 0.25, 1.0]
                list_elements = [s.strip() for s in value_str[1:-1].split(',')]
                parsed_list = []
                for elem in list_elements:
                    if elem.lower() == 'true': parsed_list.append(True)
                    elif elem.lower() == 'false': parsed_list.append(False)
                    elif elem == '+inf': parsed_list.append(float('inf'))
                    elif re.match(r"^-?\d+\.?\d*(?:e[+-]?\d+)?$", elem): # Check if it's a number
                        parsed_list.append(float(elem))
                    else: parsed_list.append(elem.replace('"', '')) # Handle strings in list if any (like texture matrix)
                polygon_data[key] = parsed_list
            elif value_str.startswith('"') and value_str.endswith('"'):
                polygon_data[key] = value_str[1:-1] # String
            elif value_str.lower() == 'true':
                polygon_data[key] = True
            elif value_str.lower() == 'false':
                polygon_data[key] = False
            elif value_str == '+inf':
                polygon_data[key] = float('inf')
            elif re.match(r"^-?\d+\.?\d*(?:e[+-]?\d+)?$", value_str): # Number (int or float)
                if '.' in value_str or 'e' in value_str:
                    polygon_data[key] = float(value_str)
                else:
                    polygon_data[key] = int(value_str)
            else:
                polygon_data[key] = value_str # Fallback as string

        parsed_data['Polygons'].append(polygon_data)

    return parsed_data

# --- Main execution logic ---
# Ensure phz_files_found and extract_dir are available from previous cells
if 'phz_files_found' in locals() and phz_files_found:
    first_phz_file = phz_files_found[0] # Get the first .phz file
    extract_dir = os.path.join('/content', os.path.splitext(os.path.basename(first_phz_file))[0] + '_extracted')

    scene_phn_path = os.path.join(extract_dir, 'scene.phn')
    thumb_png_path = os.path.join(extract_dir, 'thumb.png')
    checksums_txt_path = os.path.join(extract_dir, 'checksums.txt')

    print(f"Attempting to parse: {scene_phn_path}")
    if os.path.exists(scene_phn_path):
        algodoo_data = parse_algodoo_phn(scene_phn_path)
        print("\n--- Parsed Algodoo Scene Data (FileInfo) ---")
        display(algodoo_data.get('FileVersion', 'N/A'))
        display(algodoo_data['FileInfo'])
        print("\n--- First Polygon Object (sample properties) ---")
        if algodoo_data['Polygons']:
            # Display only a subset of properties for the first polygon to avoid clutter
            sample_polygon = algodoo_data['Polygons'][0]
            display({k: sample_polygon[k] for k in ['texture', 'materialName', 'pos', 'angle', 'density', 'surfaces'] if k in sample_polygon})
            print(f"Total polygons found: {len(algodoo_data['Polygons'])}")
        else:
            print("No polygon objects found.")
    else:
        print(f"Error: 'scene.phn' not found in '{extract_dir}'. Please ensure extraction was successful.")

    print("\n--- Thumbnail Image ---")
    if os.path.exists(thumb_png_path):
        display(Image(filename=thumb_png_path))
    else:
        print(f"No 'thumb.png' found in '{extract_dir}'.")

    print("\n--- Checksums File ---")
    if os.path.exists(checksums_txt_path):
        print(f"Content of '{checksums_txt_path}':")
        with open(checksums_txt_path, 'r') as f:
            print(f.read())
        print("Note: The format of checksums.txt is proprietary and cannot be easily interpreted without specific knowledge of Algodoo's hashing mechanism.")
    else:
        print(f"No 'checksums.txt' found in '{extract_dir}'.")
else:
    print("No .phz files were found and extracted in the previous step, or 'phz_files_found' variable is not defined.")

In [ ]:
import os
import shutil

# Directory to clean up
cleanup_dir = '/content/'

print(f"Cleaning up directory: {cleanup_dir}")

# Iterate through items in the directory
for item_name in os.listdir(cleanup_dir):
    item_path = os.path.join(cleanup_dir, item_name)

    # Check if it's a .phz file
    if item_name.endswith('.phz') and os.path.isfile(item_path):
        try:
            os.remove(item_path)
            print(f"Removed file: {item_path}")
        except Exception as e:
            print(f"Error removing file {item_path}: {e}")

    # Check if it's an _extracted folder
    elif item_name.endswith('_extracted') and os.path.isdir(item_path):
        try:
            shutil.rmtree(item_path)
            print(f"Removed directory: {item_path}")
        except Exception as e:
            print(f"Error removing directory {item_path}: {e}")

print("Cleanup complete.")

Cleaning up directory: /content/
Removed file: /content/glass vase v1.phz
Removed file: /content/wooden vase v2.phz
Removed file: /content/wooden vase v1.phz
Removed file: /content/glass vase v2.phz
Cleanup complete.


In [ ]:
from google.colab import drive
drive.flush_and_unmount()
print('Drive unmounted.')

Drive unmounted.
